In [0]:
# ============================================================
# PARAMÈTRES
# ============================================================
dbutils.widgets.text("catalog",        "banking")
dbutils.widgets.text("schema_bronze",  "bronze")
dbutils.widgets.text("schema_silver",  "silver")

catalog       = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")

# Tables
source_table   = f"{catalog}.{schema_bronze}.bronze_accounts"
target_silver  = f"{catalog}.{schema_silver}.silver_accounts"
monitoring_table = f"{catalog}.{schema_bronze}.execution_monitoring"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable



# ── Transformations
df = spark.table(source_table)
print('Lignes before transformations :', df.count())

df = df.filter(F.col('account_id').isNotNull()) \
       .filter(F.col('customer_id').isNotNull())

window = Window.partitionBy('account_id').orderBy(F.col('updated_at').desc())
df = df.withColumn("row_number", F.row_number().over(window)) \
       .filter(F.col('row_number') == 1) \
       .drop('row_number') \
       .withColumn("ingestion_date", F.current_timestamp())

print('Lignes after transformations :', df.count())

# ── Watermark
try:
    last_wm = spark.sql(f"""
        SELECT COALESCE(MAX(updated_at), '1900-01-01')
        FROM {target_silver}
    """).collect()[0][0]
except:
    last_wm = '1900-01-01'

print(f"Watermark : {last_wm}")

# ── Filtrer seulement les nouvelles lignes
df = df.filter(F.col("updated_at") > last_wm)
print(f"Lignes après filtre watermark : {df.count()}")

# ── Écriture
table_exists = spark.catalog.tableExists(target_silver)

if not table_exists:
    print("Premier run — création de la table...")
    df.write.format("delta").mode("overwrite").saveAsTable(target_silver)
    print("✅ Table silver_accounts créée")
else:
    print("Merge en cours...")
    DeltaTable.forName(spark, target_silver).alias("t") \
        .merge(df.alias("n"), "t.account_id = n.account_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("✅ Merge silver_accounts terminé")